In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score

print("Libraries imported successfully!")

Libraries imported successfully!


In [3]:
from io import StringIO

raw = """id,journal_text,ambience_type,duration_min,sleep_hours,energy_level,stress_level,time_of_day,previous_day_mood,face_emotion_hint,reflection_quality,emotional_state,intensity
1,The ocean ambience helped me stop drifting and concentrate on my next steps. My to-do list feels less chaotic.,ocean,12,6.5,4,2,afternoon,mixed,calm_face,clear,focused,3
2,I tried to relax during the forest ambience yet my thoughts kept racing. I still feel a low buzz in my body.,forest,35,6,2,4,evening,calm,tired_face,vague,restless,3
3,The forest session slowed my thoughts and I feel more settled now.,forest,3,,2,1,night,overwhelmed,happy_face,clear,calm,3
4,the mountain ambience was pleasant though i can't say it shifted my mood much. idk.,mountain,25,7,4,4,night,focused,calm_face,vague,neutral,1
5,The rain session gave me a pause but the pressure is still sitting hard on me. I'm carrying too much in my head.,rain,25,5,3,5,afternoon,,tense_face,clear,overwhelmed,5
6,after the forest track i feel peaceful and less pulled in every direction. my shoulders feel less tense.,forest,12,8,3,2,morning,mixed,calm_face,vague,calm,3
7,Nothing strong came up during the rain session; I feel fairly normal. At least I paused for a moment.,rain,20,6.5,2,4,early_morning,calm,neutral_face,conflicted,neutral,1
8,even with the mountain session my mind kept jumping between tasks.,mountain,12,6,3,4,morning,neutral,tense_face,clear,restless,4
9,I couldn't really settle into the cafe track; I kept thinking of everything at once. I still feel a low buzz in my body.,cafe,8,5.5,3,4,early_morning,mixed,neutral_face,vague,restless,4
10,The mountain ambience helped me stop drifting and concentrate on my next steps I should begin with the hardest task first,mountain,15,7,4,2,morning,overwhelmed,calm_face,conflicted,focused,3
11,The rain sounds were nice but I still feel unsettled and fidgety. Part of me wants to do everything at once.,rain,12,6.5,3,3,afternoon,mixed,none,conflicted,restless,4
12,I feel mentally clear after the mountain session and ready to tackle one thing at a time.,mountain,12,5.5,3,2,morning,restless,neutral_face,clear,focused,4
13,The forest session made me calmer but part of me still feels uneasy. Part of me wants rest part of me wants action.,forest,20,5,2,2,afternoon,neutral,none,conflicted,mixed,2
14,The cafe session helped a little though I still feel pulled in too many directions. Part of me wants to do everything at once.,cafe,12,6,3,5,early_morning,mixed,none,clear,restless,4
15,I feel lighter after the mountain sounds like my mind finally softened. I think I can start gently today.,mountain,8,8,3,3,night,overwhelmed,calm_face,clear,calm,2
16,I came in distracted but I left the forest session with a sharper mind. It feels easier to make a plan now.,forest,15,,3,1,early_morning,neutral,neutral_face,vague,focused,2
17,The forest session made me calmer but part of me still feels uneasy. I feel better and not better at the same time. I can't tell if I need rest or momentum.,forest,35,6,2,3,night,,calm_face,vague,mixed,3
18,The mountain session gave me a pause but the pressure is still sitting hard on me. I'm carrying too much in my head.,mountain,15,5.5,3,5,early_morning,overwhelmed,tired_face,vague,overwhelmed,4
19,I feel mentally clear after the mountain session and ready to tackle one thing at a time. I should begin with the hardest task first.,mountain,8,7.5,3,2,night,focused,neutral_face,clear,focused,4
20,I started scattered but the ocean session helped me lock in on what matters. My to-do list feels less chaotic. idk.,ocean,15,7,4,2,afternoon,calm,tense_face,clear,focused,2
21,The mountain session made me calmer but part of me still feels uneasy. Part of me wants rest part of me wants action.,mountain,15,5,2,2,afternoon,calm,happy_face,clear,mixed,3
22,The cafe session wasn't enough today; everything still feels heavy and too much. It's hard to know where to begin.,cafe,12,,2,4,afternoon,focused,tired_face,conflicted,overwhelmed,4
23,even after the forest track i feel exhausted and emotionally overloaded i feel emotionally tired i almost wanted to stop midway,forest,18,5.5,2,4,morning,focused,tense_face,vague,overwhelmed,5
24,I liked the ocean session but my mood is still split between calm and tension. It's like two moods are sitting together. I can't tell if I need rest or momentum. idk.,ocean,20,7,4,4,afternoon,focused,tense_face,clear,mixed,4
25,Even after the mountain track I feel exhausted and emotionally overloaded. I feel emotionally tired.,mountain,12,3.5,2,5,night,mixed,tense_face,vague,overwhelmed,4
26,The forest session was okay I don't feel much different just a bit more aware Maybe I need more time to notice a difference,forest,10,7.5,3,3,night,overwhelmed,calm_face,clear,neutral,2
27,I started scattered but the cafe session helped me lock in on what matters. My to-do list feels less chaotic.,cafe,25,7.5,4,1,night,overwhelmed,neutral_face,clear,focused,4
28,The cafe ambience helped me breathe slower and let go of some pressure.,cafe,8,6.5,4,1,early_morning,neutral,none,clear,calm,3
29,The mountain background made it easier to organize my thoughts and work plan.,mountain,25,7.5,5,2,night,restless,happy_face,clear,focused,4
30,I wasn't expecting much but the rain session made me feel quiet inside My shoulders feel less tense,rain,25,6,4,2,early_morning,focused,neutral_face,clear,calm,2
31,I wanted the ocean to calm me but today my stress feels bigger than the session. I almost wanted to stop midway.,ocean,18,6,2,5,afternoon,mixed,none,conflicted,overwhelmed,5
32,I wasn't expecting much but the mountain session made me feel quiet inside. I think I can start gently today.,mountain,10,8,3,2,evening,neutral,calm_face,vague,calm,4
33,I feel mentally clear after the cafe session and ready to tackle one thing at a time.,cafe,25,7,4,2,morning,calm,neutral_face,clear,focused,2
34,The ocean session was okay. I don't feel much different just a bit more aware. Nothing really clicked yet.,ocean,25,6,4,4,night,restless,neutral_face,clear,neutral,2
35,The rain ambience helped me stop drifting and concentrate on my next steps. I should use this window well.,rain,15,,4,1,early_morning,mixed,calm_face,vague,focused,3
36,The forest session was okay. I don't feel much different just a bit more aware. Nothing really clicked yet.,forest,12,6.5,4,3,afternoon,neutral,none,clear,neutral,2
37,I feel both comforted and distracted after the cafe ambience. It's like two moods are sitting together. I can't tell if I need rest or momentum.,cafe,25,5,2,5,evening,focused,neutral_face,vague,mixed,4
38,I wanted the mountain to calm me but today my stress feels bigger than the session. I feel emotionally tired.,mountain,10,5,1,5,early_morning,,none,conflicted,overwhelmed,5
39,I feel lighter after the cafe sounds like my mind finally softened. My shoulders feel less tense.,cafe,20,7,4,1,evening,focused,calm_face,clear,calm,2
40,The rain track was fine. I feel steady not especially better or worse.,rain,20,7.5,3,2,afternoon,overwhelmed,neutral_face,clear,neutral,1
41,The rain ambience was pleasant though I can't say it shifted my mood much. Maybe I need more time to notice a difference.,rain,8,7.5,3,2,morning,overwhelmed,calm_face,conflicted,neutral,2
42,I feel both comforted and distracted after the mountain ambience There is relief but also some lingering pressure,mountain,15,5.5,3,3,early_morning,restless,neutral_face,conflicted,mixed,4
43,The mountain ambience helped me stop drifting and concentrate on my next steps. I should begin with the hardest task first.,mountain,8,7,3,2,evening,calm,none,vague,focused,2
44,I tried to relax during the mountain ambience yet my thoughts kept racing. I keep wanting to switch tasks.,mountain,18,4.5,4,5,early_morning,calm,neutral_face,conflicted,restless,3
45,The cafe sounds were nice but I still feel unsettled and fidgety. I keep wanting to switch tasks.,cafe,10,5,3,4,evening,mixed,happy_face,clear,restless,4
46,The mountain track helped a little though something still feels off underneath. There is relief but also some lingering pressure.,mountain,20,7,4,2,morning,neutral,none,clear,mixed,2
47,The cafe session slowed my thoughts and I feel more settled now. I think I can start gently today.,cafe,12,7,2,2,afternoon,focused,happy_face,clear,calm,4
48,I noticed the ocean sounds but emotionally I still feel mostly the same. Maybe I need more time to notice a difference.,ocean,18,7.5,3,3,morning,neutral,none,clear,neutral,1
49,i feel lighter after the ocean sounds like my mind finally softened i think i can start gently today,ocean,25,7.5,4,1,morning,mixed,none,clear,calm,4
50,Even with the cafe session my mind kept jumping between tasks. I still feel a low buzz in my body.,cafe,15,5.5,4,5,evening,focused,tense_face,clear,restless,4
51,Nothing strong came up during the forest session; I feel fairly normal At least I paused for a moment,forest,3,6.5,3,2,early_morning,mixed,,clear,neutral,2
52,I liked the ocean session but my mood is still split between calm and tension. Part of me wants rest part of me wants action.,ocean,12,6,3,3,evening,restless,calm_face,conflicted,mixed,4
53,I feel lighter after the ocean sounds like my mind finally softened. The pace of my breathing changed.,ocean,10,7.5,4,2,morning,,neutral_face,conflicted,calm,4
54,The ocean session wasn't enough today; everything still feels heavy and too much.,ocean,25,4.5,2,4,night,focused,,vague,overwhelmed,5
55,I feel lighter after the cafe sounds like my mind finally softened. My shoulders feel less tense.,cafe,15,6,4,2,afternoon,overwhelmed,happy_face,vague,calm,4
56,After the forest track I feel peaceful and less pulled in every direction.,forest,20,6.5,4,2,afternoon,neutral,calm_face,clear,calm,3
57,I feel lighter after the rain sounds like my mind finally softened.,rain,20,6,3,2,morning,neutral,none,clear,calm,3
58,The forest sounds were nice but I still feel unsettled and fidgety.,forest,12,6,3,3,early_morning,overwhelmed,tired_face,clear,restless,5
59,The mountain ambience helped me stop drifting and concentrate on my next steps.,mountain,8,7,3,3,afternoon,mixed,calm_face,clear,focused,3
60,The mountain ambience was pleasant though I can't say it shifted my mood much. I can continue the day as usual.,mountain,25,7,3,3,morning,restless,happy_face,clear,neutral,3
61,Even with the ocean session my mind kept jumping between tasks.,ocean,12,5,3,4,night,neutral,neutral_face,conflicted,restless,3
62,After the rain sounds I feel better than before but not completely okay.,rain,15,5.5,3,4,early_morning,neutral,none,conflicted,mixed,4
63,The mountain session wasn't enough today; everything still feels heavy and too much. Even small tasks feel big right now.,mountain,18,4.5,1,5,evening,restless,none,clear,overwhelmed,5
64,The ocean background made it easier to organize my thoughts and work plan. I can see my priorities more clearly.,ocean,20,,4,1,early_morning,focused,happy_face,conflicted,focused,2
65,after the mountain sounds i feel better than before but not completely okay. it's like two moods are sitting together.,mountain,20,4.5,3,4,afternoon,,neutral_face,vague,mixed,2
66,After the mountain track I feel peaceful and less pulled in every direction. My shoulders feel less tense.,mountain,12,8,2,2,early_morning,restless,neutral_face,clear,calm,4
67,The cafe session helped a little though I still feel pulled in too many directions.,cafe,20,5,4,5,morning,neutral,tense_face,clear,restless,4
68,The cafe background made it easier to organize my thoughts and work plan.,cafe,18,,3,3,early_morning,neutral,neutral_face,conflicted,focused,4
69,I sat through the cafe ambience but I still feel flooded by what I need to do. I almost wanted to stop midway.,cafe,3,4.5,2,5,night,restless,tired_face,clear,overwhelmed,5
70,I feel lighter after the forest sounds like my mind finally softened My shoulders feel less tense,forest,10,7,3,2,afternoon,neutral,neutral_face,clear,calm,3
71,I sat through the cafe ambience but I still feel flooded by what I need to do. I'm carrying too much in my head.,cafe,8,6,2,4,afternoon,neutral,tense_face,clear,overwhelmed,5
72,i sat through the mountain ambience but i still feel flooded by what i need to do.,mountain,18,3.5,3,5,night,neutral,tense_face,clear,overwhelmed,4
73,I noticed the ocean sounds but emotionally I still feel mostly the same At least I paused for a moment,ocean,12,5.5,2,3,morning,calm,neutral_face,clear,neutral,1
74,Nothing strong came up during the forest session; I feel fairly normal I can continue the day as usual,forest,12,7,3,4,night,restless,neutral_face,clear,neutral,1
75,I couldn't really settle into the rain track; I kept thinking of everything at once. I keep wanting to switch tasks.,rain,20,4.5,4,3,evening,overwhelmed,tense_face,vague,restless,4
76,The cafe ambience was pleasant though I can't say it shifted my mood much. At least I paused for a moment.,cafe,30,6.5,4,3,night,restless,neutral_face,conflicted,neutral,2
77,After the forest sounds I feel better than before but not completely okay. I can't tell if I need rest or momentum.,forest,18,4.5,2,2,evening,restless,calm_face,clear,mixed,3
78,After the cafe sounds I feel better than before but not completely okay. I feel better and not better at the same time.,cafe,8,6.5,3,3,evening,calm,tense_face,clear,mixed,3
79,I feel both comforted and distracted after the mountain ambience. It's like two moods are sitting together.,mountain,12,6,4,2,morning,restless,neutral_face,vague,mixed,3
80,the cafe ambience helped me stop drifting and concentrate on my next steps. i should begin with the hardest task first.,cafe,25,5.5,5,2,night,mixed,none,conflicted,focused,3
81,I feel lighter after the mountain sounds like my mind finally softened. I think I can start gently today.,mountain,12,,3,2,morning,focused,neutral_face,clear,calm,3
82,I came in distracted but I left the rain session with a sharper mind. I should begin with the hardest task first.,rain,25,6,5,2,evening,calm,neutral_face,clear,focused,4
83,I started scattered but the ocean session helped me lock in on what matters. My to-do list feels less chaotic.,ocean,15,7.5,3,2,afternoon,calm,neutral_face,clear,focused,4
84,I feel lighter after the ocean sounds like my mind finally softened. The pace of my breathing changed.,ocean,35,8,3,3,early_morning,focused,calm_face,conflicted,calm,3
85,Even after the forest track I feel exhausted and emotionally overloaded.,forest,8,5.5,2,5,afternoon,neutral,tired_face,clear,overwhelmed,5
86,The forest session slowed my thoughts and I feel more settled now. My shoulders feel less tense.,forest,10,6.5,4,3,evening,calm,,clear,calm,2
87,The forest session made me calmer but part of me still feels uneasy There is relief but also some lingering pressure,forest,8,4.5,4,4,night,focused,tense_face,clear,mixed,3
88,I noticed the rain sounds but emotionally I still feel mostly the same. Nothing really clicked yet. Maybe later I'll understand it more.,rain,20,6,3,3,evening,calm,neutral_face,vague,neutral,1
89,Even with the mountain session my mind kept jumping between tasks.,mountain,15,6.5,3,4,morning,calm,tense_face,clear,restless,4
90,I couldn't really settle into the cafe track; I kept thinking of everything at once. Part of me wants to do everything at once.,cafe,8,6.5,2,5,early_morning,overwhelmed,neutral_face,conflicted,restless,4
91,The cafe track was fine. I feel steady not especially better or worse.,cafe,20,7,3,3,evening,neutral,neutral_face,clear,neutral,1
92,The forest track was fine I feel steady not especially better or worse At least I paused for a moment Maybe later I'll understand it more,forest,15,5.5,2,3,morning,overwhelmed,neutral_face,vague,neutral,1
93,Even with the cafe session my mind kept jumping between tasks. Part of me wants to do everything at once.,cafe,10,5,3,5,early_morning,restless,tired_face,conflicted,restless,4
94,I tried to relax during the rain ambience yet my thoughts kept racing. I keep wanting to switch tasks.,rain,20,6,3,4,early_morning,overwhelmed,none,conflicted,restless,4
95,I noticed the rain sounds but emotionally I still feel mostly the same. idk.,rain,30,6.5,3,3,night,overwhelmed,neutral_face,conflicted,neutral,2
96,I couldn't really settle into the forest track; I kept thinking of everything at once. I keep wanting to switch tasks.,forest,35,6,3,3,afternoon,restless,neutral_face,clear,restless,4
97,The mountain session made me calmer but part of me still feels uneasy. I feel better and not better at the same time.,mountain,15,6,3,3,afternoon,neutral,neutral_face,clear,mixed,2
98,The ocean session helped a little though I still feel pulled in too many directions. I still feel a low buzz in my body.,ocean,20,4.5,2,5,afternoon,mixed,neutral_face,clear,restless,4
99,I feel mentally clear after the rain session and ready to tackle one thing at a time. I can see my priorities more clearly. I should use this window well.,rain,18,6,4,1,evening,overwhelmed,neutral_face,clear,focused,2"""

train_df = pd.read_csv(StringIO(raw))
print("Training data loaded:", train_df.shape)
print(train_df['emotional_state'].value_counts())

Training data loaded: (99, 13)
emotional_state
focused        18
restless       18
calm           18
neutral        17
mixed          15
overwhelmed    13
Name: count, dtype: int64


In [23]:
# CELL 3 - Load Test Data
test_df = pd.read_excel('data/arvyax_test_inputs_120.xlsx')
print("Test data loaded:", test_df.shape)
print(test_df.head(3))
print(test_df.isnull().sum())

Test data loaded: (120, 11)
      id                                       journal_text ambience_type  \
0  10001  woke up feeling more organized mentally. i was...          cafe   
1  10002  started off distracted most of the time. this ...      mountain   
2  10003                                     kinda calm ...          cafe   

   duration_min  sleep_hours  energy_level  stress_level time_of_day  \
0             4          8.5             3             1       night   
1             4          8.5             1             2   afternoon   
2            15          8.5             2             5     evening   

  previous_day_mood face_emotion_hint reflection_quality  
0             mixed        happy_face              vague  
1             mixed        happy_face              clear  
2              calm        happy_face              vague  
id                     0
journal_text           0
ambience_type          0
duration_min           0
sleep_hours            0
energy_level 

In [24]:
# Training data
train_df['sleep_hours'] = train_df['sleep_hours'].fillna(train_df['sleep_hours'].median())
train_df['previous_day_mood'] = train_df['previous_day_mood'].fillna('unknown')
train_df['face_emotion_hint'] = train_df['face_emotion_hint'].fillna('none')

# Test data
test_df['sleep_hours'] = test_df['sleep_hours'].fillna(train_df['sleep_hours'].median())
test_df['previous_day_mood'] = test_df['previous_day_mood'].fillna('unknown')
test_df['face_emotion_hint'] = test_df['face_emotion_hint'].fillna('none')

print(" Missing values filled!")
print("Train nulls:", train_df.isnull().sum().sum())
print("Test nulls:", test_df.isnull().sum().sum())

 Missing values filled!
Train nulls: 0
Test nulls: 0


In [9]:
def extract_text_features(text):
    text = str(text).lower()
    features = {}
    features['text_len'] = len(text)
    features['word_count'] = len(text.split())
    features['calm_words']       = sum(1 for w in ['calm','peaceful','settled','lighter','softened','quiet','gentle'] if w in text)
    features['restless_words']   = sum(1 for w in ['racing','jumping','scattered','fidgety','unsettled','buzz','switch','everything at once'] if w in text)
    features['focused_words']    = sum(1 for w in ['clear','concentrate','plan','priorities','lock in','sharper','task','work'] if w in text)
    features['overwhelmed_words']= sum(1 for w in ['heavy','flooded','carrying','overloaded','exhausted','pressure','too much','stress'] if w in text)
    features['mixed_words']      = sum(1 for w in ['both','split','two moods','better and not','rest or momentum','uneasy','conflicted'] if w in text)
    features['neutral_words']    = sum(1 for w in ['okay','fine','normal','same','nothing','aware','steady'] if w in text)
    features['uncertainty']      = sum(1 for w in ['idk',"can't tell",'maybe','not sure'] if w in text)
    return features


train_text_feats = train_df['journal_text'].apply(extract_text_features).apply(pd.Series)
test_text_feats  = test_df['journal_text'].apply(extract_text_features).apply(pd.Series)

train_df = pd.concat([train_df.reset_index(drop=True), train_text_feats], axis=1)
test_df  = pd.concat([test_df.reset_index(drop=True),  test_text_feats],  axis=1)

print("Text features extracted!")
print("New columns added:", list(train_text_feats.columns))

Text features extracted!
New columns added: ['text_len', 'word_count', 'calm_words', 'restless_words', 'focused_words', 'overwhelmed_words', 'mixed_words', 'neutral_words', 'uncertainty']


In [10]:
cat_cols = ['ambience_type','time_of_day','previous_day_mood','face_emotion_hint','reflection_quality']
le_dict = {}

for col in cat_cols:
    le = LabelEncoder()
    # Fit on combined data so test unseen values are handled
    combined = pd.concat([train_df[col], test_df[col]]).astype(str)
    le.fit(combined)
    train_df[col+'_enc'] = le.transform(train_df[col].astype(str))
    test_df[col+'_enc']  = le.transform(test_df[col].astype(str))
    le_dict[col] = le

print("Categorical columns encoded!")

Categorical columns encoded!


In [11]:
meta_features = ['duration_min','sleep_hours','energy_level','stress_level',
                 'ambience_type_enc','time_of_day_enc','previous_day_mood_enc',
                 'face_emotion_hint_enc','reflection_quality_enc',
                 'text_len','word_count','calm_words','restless_words',
                 'focused_words','overwhelmed_words','mixed_words','neutral_words','uncertainty']

# TF-IDF
tfidf = TfidfVectorizer(max_features=50, ngram_range=(1,2), stop_words='english')
X_train_tfidf = tfidf.fit_transform(train_df['journal_text']).toarray()
X_test_tfidf  = tfidf.transform(test_df['journal_text']).toarray()

# Final feature matrices
X_train = np.hstack([X_train_tfidf, train_df[meta_features].values])
X_test  = np.hstack([X_test_tfidf,  test_df[meta_features].values])

# Labels
le_state = LabelEncoder()
y_state     = le_state.fit_transform(train_df['emotional_state'])
y_intensity = train_df['intensity'].values

print("Feature matrices ready!")
print("X_train shape:", X_train.shape)
print("X_test shape: ", X_test.shape)

Feature matrices ready!
X_train shape: (99, 68)
X_test shape:  (120, 68)


In [12]:
# Emotional State Model
clf_state = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42, class_weight='balanced')
clf_state.fit(X_train, y_state)

# Intensity Model  
clf_intensity = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42)
clf_intensity.fit(X_train, y_intensity)

# Cross-validation scores
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_state = cross_val_score(clf_state, X_train, y_state, cv=cv, scoring='f1_weighted')
cv_int   = cross_val_score(clf_intensity, X_train, y_intensity, cv=cv, scoring='f1_weighted')

print(" Models trained!")
print(f"Emotional State CV F1:  {cv_state.mean():.3f} ± {cv_state.std():.3f}")
print(f"Intensity CV F1:        {cv_int.mean():.3f}   ± {cv_int.std():.3f}")

 Models trained!
Emotional State CV F1:  0.940 ± 0.046
Intensity CV F1:        0.386   ± 0.079


In [13]:
def decide_what(state, intensity, stress, energy, time_of_day):
    if state == 'overwhelmed' and intensity >= 4:
        return 'box_breathing'
    elif state == 'overwhelmed':
        return 'grounding'
    elif state == 'restless' and stress >= 4:
        return 'box_breathing'
    elif state == 'restless':
        return 'movement'
    elif state == 'focused' and energy >= 3:
        return 'deep_work'
    elif state == 'focused':
        return 'light_planning'
    elif state == 'calm' and time_of_day in ['night','evening']:
        return 'rest'
    elif state == 'calm':
        return 'journaling'
    elif state == 'mixed' and stress >= 3:
        return 'journaling'
    elif state == 'mixed':
        return 'sound_therapy'
    elif state == 'neutral' and energy >= 3:
        return 'light_planning'
    else:
        return 'pause'

def decide_when(state, intensity, time_of_day, stress):
    if state == 'overwhelmed' and intensity >= 4:
        return 'now'
    elif state == 'restless' and stress >= 4:
        return 'now'
    elif state == 'focused':
        return 'now'
    elif state == 'calm' and time_of_day in ['night','evening']:
        return 'tonight'
    elif state == 'calm':
        return 'within_15_min'
    elif time_of_day in ['night','evening'] and state in ['neutral','mixed']:
        return 'tomorrow_morning'
    elif time_of_day in ['morning','early_morning']:
        return 'within_15_min'
    else:
        return 'later_today'

print("Decision engine ready!")

Decision engine ready!


In [25]:
# CELL 10 - FINAL FIXED Confidence & Uncertainty

def get_confidence(proba, reflection_quality, word_count):
    max_prob = np.max(proba)
    sorted_p = np.sort(proba)[::-1]
    margin   = sorted_p[0] - sorted_p[1] if len(sorted_p) > 1 else 1.0

    # Base confidence = max probability directly (no normalization)
    conf = max_prob

    # Only penalize VERY short text (less than 4 words)
    if word_count < 4:
        conf *= 0.80

    # Small penalty for conflicted quality
    if reflection_quality == 'conflicted':
        conf *= 0.90

    conf = round(min(max(conf, 0.0), 1.0), 3)

    # Uncertain ONLY if model is truly split between classes
    uncertain_flag = 1 if margin < 0.05 else 0

    return conf, uncertain_flag

print("Final confidence function ready!")

Final confidence function ready!


In [21]:
state_probas = clf_state.predict_proba(X_test)
pred_states  = clf_state.predict(X_test)
pred_ints    = clf_intensity.predict(X_test)

results = []
for i, row in test_df.iterrows():
    pred_state = le_state.inverse_transform([pred_states[i]])[0]
    pred_int   = int(pred_ints[i])
    what = decide_what(pred_state, pred_int, row['stress_level'], row['energy_level'], row['time_of_day'])
    when = decide_when(pred_state, pred_int, row['time_of_day'], row['stress_level'])
    conf, uflag = get_confidence(state_probas[i], row['reflection_quality'], row['word_count'])
    
    results.append({
        'id':                  row['id'],
        'predicted_state':     pred_state,
        'predicted_intensity': pred_int,
        'confidence':          conf,
        'uncertain_flag':      uflag,
        'what_to_do':          what,
        'when_to_do':          when
    })

predictions_df = pd.DataFrame(results)
print("Predictions generated!")
print(predictions_df.head(10).to_string())

Predictions generated!
      id predicted_state  predicted_intensity  confidence  uncertain_flag      what_to_do        when_to_do
0  10001         neutral                    4       0.366               0  light_planning  tomorrow_morning
1  10002         neutral                    4       0.327               0           pause       later_today
2  10003            calm                    4       0.246               0            rest           tonight
3  10004         neutral                    4       0.414               0           pause     within_15_min
4  10005     overwhelmed                    4       0.274               1   box_breathing               now
5  10006         neutral                    4       0.220               1  light_planning     within_15_min
6  10007         neutral                    4       0.483               0           pause  tomorrow_morning
7  10008         neutral                    4       0.287               0           pause       later_today
8  10

In [22]:
predictions_df.to_csv('predictions.csv', index=False)
print(" predictions.csv saved!")
print(f"Total rows: {len(predictions_df)}")
print("\nState distribution:")
print(predictions_df['predicted_state'].value_counts())
print("\nUncertain cases:", predictions_df['uncertain_flag'].sum())

 predictions.csv saved!
Total rows: 120

State distribution:
predicted_state
neutral        83
overwhelmed     9
focused         8
mixed           8
calm            6
restless        6
Name: count, dtype: int64

Uncertain cases: 19
